# 04 — Data Quality Checks

**Purpose:** Fabric runtime quality gates and revenue reconciliation.  
**Runtime:** Microsoft Fabric PySpark with attached default Lakehouse.  
**Status:** Provided as source; requires execution and verification in your Fabric tenant.

In [ ]:
from pyspark.sql import functions as F

customers=spark.table('silver_customers')
orders=spark.table('silver_orders')
items=spark.table('silver_order_items')
products=spark.table('silver_products')
daily=spark.table('gold_daily_sales')
product_perf=spark.table('gold_product_performance')
customer360=spark.table('gold_customer_360')

assert customers.count()==customers.select('customer_id').distinct().count(), 'Duplicate customers'
assert items.count()==items.select('order_item_id').distinct().count(), 'Duplicate item IDs'
assert items.filter(F.col('quantity')<=0).count()==0, 'Nonpositive quantity'
assert items.join(products,'product_id','left_anti').count()==0, 'Orphan product IDs'
assert items.join(orders,'order_id','left_anti').count()==0, 'Orphan order IDs'
assert customer360.count()==customers.count(), 'Customer 360 excludes customers'

# Tolerance accounts for floating point aggregation of line-level rounded money.
revenue_daily=float(daily.agg(F.sum('net_revenue')).first()[0] or 0)
revenue_product=float(product_perf.agg(F.sum('net_revenue')).first()[0] or 0)
revenue_customer=float(customer360.agg(F.sum('lifetime_revenue')).first()[0] or 0)
assert abs(revenue_daily-revenue_product)<0.02, 'Daily/product revenue mismatch'
assert abs(revenue_daily-revenue_customer)<0.02, 'Daily/customer revenue mismatch'
completed=orders.filter(F.col('order_status')=='Completed')
assert int(daily.agg(F.sum('orders')).first()[0] or 0)<=completed.count(), 'Completed order count invalid'
print('PASS: uniqueness, valid references, quantities, customer coverage, Gold revenue reconciliation')
print(f'Gold total revenue: ${revenue_daily:,.2f}')
